# «Αύρα»: the wake word for AURA (microWakeWord)

Built from microWakeWord's own `basic_training_notebook.ipynb` (Apache-2.0,
github.com/kahrendt/microWakeWord) for AURA's ESP32-S3 (2026-11-19).

**Πώς το τρέχεις (Greek):**
1. Χρόνος εκτέλεσης → Αλλαγή τύπου χρόνου εκτέλεσης → **T4 GPU** (πάνω δεξιά: «T4»).
2. Προαιρετικά, για ελληνικές φωνές: 🔑 (αριστερά) → μυστικά `AZURE_SPEECH_KEY` και
   `AZURE_SPEECH_REGION`: μόνο η τιμή μετά το `=` στο `.env` του AURA, με πρόσβαση σημειωματαρίου.
3. Χρόνος εκτέλεσης → **Εκτέλεση όλων**. Μετά την εγκατάσταση σταματά στο 2ο κελί
   και ζητά επανεκκίνηση: Χρόνος εκτέλεσης → **Επανεκκίνηση συνεδρίας**.
4. Ξανά Χρόνος εκτέλεσης → **Εκτέλεση όλων**: η εγκατάσταση παραλείπεται.
   Η «Εκτέλεση όλων» σταματά στο κελί της εκπαίδευσης. Πριν από αυτό: Χρόνος εκτέλεσης →
   **Επανεκκίνηση συνεδρίας** (για να έχει όλη τη μνήμη), ▶ στο κελί που κρατά 4 ώρες από
   τον θόρυβο ελέγχου, και ▶ **μόνο** στο κελί της εκπαίδευσης. Μετά ▶ στα δύο κελιά λήψης.
5. Στο τέλος κατεβαίνουν τα `avra.tflite` (~60 KB) και `avra.json`: στείλ' τα στον Claude,
   με τις τελευταίες γραμμές του κελιού της εκπαίδευσης.

Updated 2026-11-20 for Colab's Python 3.13: piper-sample-generator 3.x runs as a
package with its phonemes from piper-tts, and datasets 4 decodes audio through
torchcodec.

The first model is a starting point: microWakeWord's authors say a usable one
usually takes some tuning (more samples, steps, the negative class weight).
Expect about 2-3 hours on a T4 and ~6 GB of downloads (Colab's disk, not yours).


In [ ]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
# AURA (2026-11-20): only while microWakeWord does not import, so the Run all
# after the restart goes straight on.
import importlib.util
import platform

if importlib.util.find_spec("microwakeword") is None:
    if platform.system() == "Darwin":
        # `pymicro-features` is installed from a fork to support building on macOS
        !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

    # `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
    !pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

    !git clone https://github.com/kahrendt/microWakeWord
    !pip install -e ./microWakeWord

    # AURA (2026-11-20): Colab runs Python 3.13. The samples' phonemes come from
    # piper-tts (piper-phonemize has no build for 3.13), and datasets 4 decodes
    # audio through torchcodec.
    !pip install piper-tts==1.3.0 torchcodec
    print("Τώρα: Χρόνος εκτέλεσης → Επανεκκίνηση συνεδρίας, μετά Εκτέλεση όλων (restart the session).")
else:
    print("microWakeWord is installed.")

In [ ]:
# Generates 1 sample of the target word for manual verification.
# AURA (2026-11-20): piper-sample-generator 3.x is a package run from its own
# folder (generate_samples.py is gone), pinned to the commit read on that day.
# Its pip package is not installed: it pins an older audiomentations than
# microWakeWord's.

import importlib.util

if importlib.util.find_spec("microwakeword") is None:
    raise RuntimeError("Χρόνος εκτέλεσης → Επανεκκίνηση συνεδρίας, και ξανά Εκτέλεση όλων "
                       "(restart the session: the install is seen only after it).")

target_word = 'Avra'  # «Αύρα», [ˈavra]; more spellings below

import os

from IPython.display import Audio

PSG = '/content/piper-sample-generator'
PSG_COMMIT = '2971426a55072f7d22fec416ca7800df8bd23207'  # master, 2026-03-12
GENERATOR = f'{PSG}/models/en_US-libritts_r-medium.pt'

if not os.path.exists(PSG):
    !git clone https://github.com/rhasspy/piper-sample-generator {PSG}
    !git -C {PSG} checkout -q {PSG_COMMIT}
if not os.path.exists(GENERATOR):
    !wget -O {GENERATOR} 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

!cd {PSG} && python3 -m piper_sample_generator "{target_word}" \
--model {GENERATOR} \
--max-samples 1 \
--batch-size 1 \
--output-dir /content/generated_samples

Audio("/content/generated_samples/0.wav", autoplay=True)


In [ ]:
# Many samples of «Αύρα» from several English spellings, each with the
# multi-speaker LibriTTS voice (904 speakers): spellings vary the vowels.
spellings = ['Avra', 'Ahvra', 'Avvra', 'Ah vra', 'Avrah', 'Ahv rah']
for index, spelling in enumerate(spellings):
    !cd {PSG} && python3 -m piper_sample_generator "{spelling}" \
    --model {GENERATOR} \
    --max-samples 1500 \
    --batch-size 100 \
    --noise-scales 0.667 0.85 \
    --noise-scale-ws 0.8 1.0 \
    --length-scales 0.85 1.0 1.15 \
    --output-dir /content/generated_samples/spelling_{index}
!mkdir -p generated_samples/all && find generated_samples -name '*.wav' -not -path '*/all/*' -exec sh -c 'cp "$1" "generated_samples/all/$(echo $1 | tr / _)"' _ {} \;
!ls generated_samples/all | wc -l


In [ ]:
# Optional: «Αύρα» in real Greek, from Azure's Greek and multilingual neural
# voices (the AZURE_SPEECH_KEY / AZURE_SPEECH_REGION Colab secrets; skipped
# without them). About 4 characters a sample: well within Azure's free tier.
import os, random, time, requests
try:
    from google.colab import userdata
    azure_key = userdata.get('AZURE_SPEECH_KEY')
    azure_region = userdata.get('AZURE_SPEECH_REGION')
except Exception:
    azure_key = azure_region = None

voices = ['el-GR-AthinaNeural', 'el-GR-NestorasNeural',
          'en-US-AvaMultilingualNeural', 'en-US-AndrewMultilingualNeural',
          'en-US-EmmaMultilingualNeural', 'en-US-BrianMultilingualNeural',
          'de-DE-SeraphinaMultilingualNeural', 'fr-FR-VivienneMultilingualNeural',
          'it-IT-GiuseppeMultilingualNeural', 'es-ES-ArabellaMultilingualNeural']
texts = ['Αύρα', 'Αύρα!', 'Αύρα;', 'Αύρα,']
os.makedirs('generated_samples/all', exist_ok=True)
made = 0
if azure_key and azure_region:
    url = f'https://{azure_region}.tts.speech.microsoft.com/cognitiveservices/v1'
    headers = {'Ocp-Apim-Subscription-Key': azure_key, 'Content-Type': 'application/ssml+xml',
               'X-Microsoft-OutputFormat': 'riff-16khz-16bit-mono-pcm', 'User-Agent': 'aura-wakeword'}
    for voice in voices:
        for rate in ('-20%', '-10%', '0%', '+10%', '+20%'):
            for pitch in ('-10%', '0%', '+10%'):
                text = random.choice(texts)
                ssml = (f"<speak version='1.0' xml:lang='el-GR'><voice name='{voice}'><lang xml:lang='el-GR'>"
                        f"<prosody rate='{rate}' pitch='{pitch}'>{text}</prosody></lang></voice></speak>")
                answer = requests.post(url, headers=headers, data=ssml.encode('utf-8'), timeout=30)
                if answer.status_code == 200:
                    with open(f'generated_samples/all/azure_{voice}_{rate}_{pitch}.wav', 'wb') as out:
                        out.write(answer.content)
                    made += 1
                elif answer.status_code == 429:
                    time.sleep(2)
    print('Azure samples:', made)
else:
    print('No Azure secrets: English spellings only.')


In [ ]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.
#
# AURA (2026-11-20): datasets 4 decodes audio through torchcodec and gives no
# clip path any more: the files are named from their own paths, or numbered.


import datasets
import scipy
import os

import numpy as np

from pathlib import Path
from tqdm import tqdm

## Download MIR RIR data

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    rir_dataset = rir_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    # Save clips to 16-bit PCM wav files
    for index, row in enumerate(tqdm(rir_dataset)):
        name = f"rir_{index:04d}.wav"
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

# AURA (2026-11-20): AudioSet is parquet now (its .tar files give 404): four
# shards of the balanced set, 2,000 clips of 10 s (500 a shard).
if not os.path.exists("audioset"):
    os.mkdir("audioset")
    shards = ["06", "07", "08", "09"]
    for shard in shards:
        !wget -O audioset/bal_train{shard}.parquet https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/{shard}.parquet

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files; a clip that cannot be decoded is skipped
    audioset_dataset = datasets.Dataset.from_parquet([f"audioset/bal_train{shard}.parquet" for shard in shards])
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for index in tqdm(range(len(audioset_dataset))):
        try:
            samples = audioset_dataset[index]['audio']['array']
        except Exception:
            continue
        scipy.io.wavfile.write(os.path.join(output_dir, f"audioset_{index:05d}.wav"), 16000,
                               (np.clip(samples, -1, 1)*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    !wget -O {out_dir} {link}
    !cd {output_dir} && unzip -q {fname}

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    fma_paths = [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]
    fma_dataset = datasets.Dataset.from_dict({"audio": fma_paths})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for clip_path, row in zip(fma_paths, tqdm(fma_dataset)):
        name = clip_path.split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


In [ ]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

# AURA (2026-11-20): piper-sample-generator 3.x keeps the silence around each
# sample (2.x trimmed it), so the clips are trimmed here with webrtcvad.
clips = Clips(input_directory='generated_samples/all',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=True,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )


In [ ]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

In [ ]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname

        zip_path = f"negative_datasets/{fname}"
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

In [ ]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
# AURA (2026-11-20): 10,000 as microWakeWord's own notebook; 20,000 took about
# 3 hours with validation, past what a free Colab runtime may last.
config["training_steps"] = [10000]

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    1000  # Test the validation sets after every this many steps (AURA: 500 → 1000)
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)


In [ ]:
# AURA (2026-11-20): free Colab's 12.7 GB RAM cannot hold the validation's
# ambient set (4 dinner-party recordings, 9.7 hours, cut into 116,000 windows of
# 2 s: 3.8 GB as float32, copied two or three times), so the training was cut
# (^C) at its first validation. Each recording keeps its start, 4 hours in all;
# the full set moves to negative_datasets_aside/ (nothing is deleted).
import shutil
from pathlib import Path

from mmap_ninja.ragged import RaggedMmap

KEEP_HOURS = 4.0
FRAMES_AN_HOUR = 360000  # a feature window every 10 ms
folder = Path("negative_datasets/dinner_party_eval/validation_ambient")
aside = Path("negative_datasets_aside/dinner_party_eval/validation_ambient")
if folder.exists() and not aside.exists():
    aside.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(folder), str(aside))
    recordings = [RaggedMmap(str(p)) for p in sorted(aside.glob("**/*_mmap"))]
    frames_each = int(KEEP_HOURS * FRAMES_AN_HOUR / sum(len(r) for r in recordings))
    folder.mkdir(parents=True)
    RaggedMmap.from_generator(out_dir=str(folder / "subset_mmap"),
                              sample_generator=(r[i][:frames_each] for r in recordings for i in range(len(r))),
                              batch_size=1, verbose=False)
kept = 0
for path in folder.glob("**/*_mmap"):
    subset = RaggedMmap(str(path))
    kept += sum(subset[i].shape[0] for i in range(len(subset)))
print(f"validation_ambient: {kept / FRAMES_AN_HOUR:.1f} hours kept")

In [ ]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.
#
# AURA (2026-11-20): free Colab has 12.7 GB of RAM and the cells before keep
# some in the session; the training was cut (^C) at its first validation until
# the cell above cut the ambient set to 4 hours. Before this cell: Runtime →
# Restart session, then run this cell alone (it needs only the files on disk).

import psutil

if psutil.virtual_memory().available < 8 * 1024**3:
    raise RuntimeError("Χρόνος εκτέλεσης → Επανεκκίνηση συνεδρίας, και ▶ μόνο σε αυτό το κελί "
                       "(the training needs ~8 GB of free RAM).")

!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# Downloads the tflite model file. To use on the device, you need to write a
# Model JSON file. See https://esphome.io/components/micro_wake_word for the
# documentation and
# https://github.com/esphome/micro-wake-word-models/tree/main/models/v2 for
# examples. Adjust the probability threshold based on the test results obtained
# after training is finished. You may also need to increase the Tensor arena
# model size if the model fails to load.

from google.colab import files

files.download(f"trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")

In [ ]:
# AURA's model files: the streaming quantized model and its manifest.
import json, shutil
shutil.copy('trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite', 'avra.tflite')
manifest = {
    'type': 'micro', 'wake_word': 'Αύρα', 'author': 'AURA (microWakeWord)', 'website': 'https://github.com/kahrendt/microWakeWord',
    'model': 'avra.tflite', 'trained_languages': ['el'], 'version': 2,
    'micro': {'probability_cutoff': 0.97, 'sliding_window_size': 5, 'feature_step_size': 10,
              'tensor_arena_size': 30000, 'minimum_esphome_version': '2024.7.0'}}
json.dump(manifest, open('avra.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
from google.colab import files
files.download('avra.tflite')
files.download('avra.json')
